# Week 7 Exercise: Fine-Tuning an Open-Source Model for Price Prediction

Train on a small subset of `ed-donner/pricer-data` using QLoRA.

This notebook is designed to run in **Google Colab** with a GPU (T4 or A100). It loads the full dataset from HuggingFace, extracts a subset for training (to fit limited GPU memory), fine-tunes Llama-3.2-1B with QLoRA, and evaluates the model.

**Requirements:**
- HuggingFace token (add as secret `HF_TOKEN` in Colab)
- Accept the [Meta Llama 3.2 license](https://huggingface.co/meta-llama/Llama-3.2-1B) on HuggingFace
- Optional: Weights & Biases API key (`WANDB_API_KEY`) for training logs

In [ ]:
# Install required packages (run once in Colab)
%pip install -q datasets transformers torch peft bitsandbytes trl accelerate
# Optional: add wandb if using LOG_TO_WANDB=True
# %pip install -q wandb

In [ ]:
# Imports
import os
import re
import math
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    set_seed,
)
from datasets import load_dataset
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score

## Configuration

Adjust these for your setup:
- **HF_USER:** Your HuggingFace username (for pushing the model)
- **TRAIN_SUBSET_SIZE:** Reduce for T4 (2k-5k), increase for A100 (10k-20k)
- **BATCH_SIZE:** 2-4 for T4, 8-16 for A100

In [ ]:
# === Configuration ===
BASE_MODEL = "meta-llama/Llama-3.2-1B"
DATASET_NAME = "ed-donner/pricer-data"
HF_USER = "mideseniordev"  # Change to your HuggingFace username

# Subset sizes (load full from HF, then sample)
TRAIN_SUBSET_SIZE = 5000
TEST_SUBSET_SIZE = 500
VAL_RATIO = 0.1
RANDOM_SEED = 42

# Sequence / training
MAX_SEQUENCE_LENGTH = 182
QUESTION = "What does this cost to the nearest dollar?"
PREFIX = "Price is $"

# QLoRA
LORA_R = 16
LORA_ALPHA = 32
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]
LORA_DROPOUT = 0.1

# Training
EPOCHS = 1
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 2
LEARNING_RATE = 5e-5
LR_SCHEDULER_TYPE = "cosine"
WARMUP_RATIO = 0.03
OPTIMIZER = "paged_adamw_32bit"
SAVE_STEPS = 500
LOG_TO_WANDB = False  # Set True if you have WANDB_API_KEY in Colab secrets

%matplotlib inline

## Log in to HuggingFace

Add your `HF_TOKEN` in Colab: **Secrets** (key icon) → New secret → `HF_TOKEN`

In [ ]:
hf_token = userdata.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

## Load Dataset and Extract Subset

Load `ed-donner/pricer-data` from HuggingFace, then sample a subset for training. No local copy needed.

In [ ]:
# Load full dataset from HuggingFace
dataset = load_dataset(DATASET_NAME)
train_full = dataset["train"]
test_full = dataset["test"]

# Extract subset (shuffle for variety, reproducible with seed)
train_subset = train_full.shuffle(seed=RANDOM_SEED).select(
    range(min(TRAIN_SUBSET_SIZE, len(train_full)))
)
test_subset = test_full.shuffle(seed=RANDOM_SEED).select(
    range(min(TEST_SUBSET_SIZE, len(test_full)))
)

# Split train into train/val
split = train_subset.train_test_split(test_size=VAL_RATIO, seed=RANDOM_SEED)
train_data = split["train"]
val_data = split["test"]

print(f"Train: {len(train_data):,} | Val: {len(val_data):,} | Test: {len(test_subset):,}")

In [ ]:
# Format data for SFT: prompt + completion (required for completion_only_loss)
# pricer-data has "text" (description) and "price"
def format_for_training(example):
    desc = example["text"]
    price = float(example["price"])
    prompt = f"{QUESTION}\n\n{desc}\n\n{PREFIX}"
    completion = f"{int(round(price))}.00"
    return {"prompt": prompt, "completion": completion, "price": price}

train_formatted = train_data.map(format_for_training)
val_formatted = val_data.map(format_for_training)

# Preview
ex = train_formatted[0]
print("Example formatted example:")
print(f"Prompt: {ex['prompt'][:120]}...")
print(f"Completion: {ex['completion']}")

## Load Model and Tokenizer

Load base model with 4-bit quantization (QLoRA) for memory efficiency.

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
base_model = prepare_model_for_kbit_training(base_model)

print(f"Model memory: {base_model.get_memory_footprint() / 1e6:.1f} MB")

## QLoRA + Training Setup

- **prompt + completion format + completion_only_loss=True:** Train only on the price tokens
- **LoraConfig:** Low-rank adapters on attention layers.
- **SFTConfig:** Supervised fine-tuning parameters.

In [ ]:
from datetime import datetime

# Detect GPU: bf16 needs Ampere+ (A100); fp16 works on T4 and older. Use CPU only as fallback.
use_gpu = torch.cuda.is_available()
use_bf16 = use_gpu and torch.cuda.get_device_capability()[0] >= 8
if not use_gpu:
    print("WARNING: No GPU detected. Enable GPU in Colab: Runtime → Change runtime type → GPU")

RUN_NAME = f"pricer-subset-{datetime.now():%Y-%m-%d_%H.%M.%S}"
OUTPUT_DIR = f"./{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{RUN_NAME}"

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,  # was max_seq_length in older trl
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    warmup_ratio=WARMUP_RATIO,
    optim=OPTIMIZER,
    weight_decay=0.001,
    max_grad_norm=0.3,
    use_cpu=not use_gpu,
    fp16=use_gpu and not use_bf16,
    bf16=use_bf16,
    logging_steps=25,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=250,
    report_to="wandb" if LOG_TO_WANDB else "none",
    completion_only_loss=True,  # train only on completion tokens
)

In [ ]:
# Optional: log in to Weights & Biases (only if LOG_TO_WANDB=True)
if LOG_TO_WANDB:
    import wandb
    wandb_api_key = userdata.get("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = wandb_api_key
    wandb.login()

## Train

In [ ]:
trainer = SFTTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    peft_config=lora_config,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Model saved to {OUTPUT_DIR}")

## Evaluation

Load the fine-tuned model and evaluate on the test subset.

In [ ]:
# Load base model + fine-tuned adapters
base_for_eval = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
fine_tuned_model = PeftModel.from_pretrained(base_for_eval, OUTPUT_DIR)
tokenizer_eval = AutoTokenizer.from_pretrained(OUTPUT_DIR)
fine_tuned_model.eval()

In [ ]:
def extract_price(response: str) -> float:
    """Extract numeric price from model output."""
    if PREFIX in response:
        tail = response.split(PREFIX)[-1]
    else:
        tail = response
    tail = tail.replace(",", "")
    match = re.search(r"[-+]?\d*\.?\d+", tail)
    return float(match.group()) if match else 0.0


def predict_price(datapoint, model, tokenizer) -> float:
    """Predict price for a single datapoint."""
    set_seed(42)
    device = next(model.parameters()).device
    prompt = f"{QUESTION}\n\n{datapoint['text']}\n\n{PREFIX}"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return extract_price(response)

In [ ]:
# Evaluate on test subset
EVAL_SIZE = min(200, len(test_subset))
predictions = []
ground_truths = []

for i in tqdm(range(EVAL_SIZE), desc="Evaluating"):
    dp = test_subset[i]
    pred = predict_price(dp, fine_tuned_model, tokenizer_eval)
    truth = float(dp["price"])
    predictions.append(pred)
    ground_truths.append(truth)

# Metrics
mae = sum(abs(p - t) for p, t in zip(predictions, ground_truths)) / len(predictions)
mse = mean_squared_error(ground_truths, predictions)
r2 = r2_score(ground_truths, predictions) * 100

print(f"MAE: ${mae:,.2f}")
print(f"MSE: {mse:,.0f}")
print(f"R²: {r2:.1f}%")

In [ ]:
# Scatter plot: predicted vs actual
plt.figure(figsize=(8, 8))
max_val = max(max(ground_truths), max(predictions), 1)
plt.scatter(ground_truths, predictions, alpha=0.5)
plt.plot([0, max_val], [0, max_val], "k--", label="Perfect prediction")
plt.xlabel("Actual price ($)")
plt.ylabel("Predicted price ($)")
plt.title(f"Fine-tuned Model: MAE=${mae:,.2f}")
plt.legend()
plt.tight_layout()
plt.show()

## Sample predictions

Spot-check a few examples.

In [ ]:
for i in range(5):
    dp = test_subset[i]
    pred = predict_price(dp, fine_tuned_model, tokenizer_eval)
    truth = float(dp["price"])
    desc = dp["text"][:60] + "..." if len(dp["text"]) > 60 else dp["text"]
    print(f"Desc: {desc}")
    print(f"  Predicted: ${pred:,.2f} | Actual: ${truth:,.2f} | Error: ${abs(pred - truth):,.2f}")
    print()

## Push to HuggingFace Hub

In [ ]:
trainer.push_to_hub(HUB_MODEL_NAME)
# Or push the adapter + config manually:
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(folder_path=OUTPUT_DIR, repo_id=HUB_MODEL_NAME, repo_type="model")